<a href="https://colab.research.google.com/github/gayathrikakumani24-droid/Artificial-Neural-Networks-ANN-/blob/main/ANN%20SmartGrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module I: Practice Exercise - 'Smart-Grid AI' (Big Arch Spec)

### **Context: The National Energy Grid**
You are the Lead Engineer for the National Grid. You have sensor readings from 100 power stations.
Your sensors measure:
1.  **Temperature (C)**
2.  **Wind Speed (km/h)**
3.  **Current Load (MW)**

You have **two separate goals**:
* **Goal A (Stability):** Predict if the grid is **'Unstable' (1)** or **'Stable' (0)**. (Classification)
* **Goal B (Demand):** Predict the **Next Hour Demand (MW)**. (Regression)

---
**INSTRUCTIONS:**
This exercise uses **Deeper and Wider Architectures**.
Pay attention to the layer dimensions (Input -> Hidden -> Output) to ensure the shapes match.

In [ ]:
# CELL 1: DATA GENERATION (Run this first)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(500)

# 100 Power Stations
N = 100
# Features: Temp (-10 to 40), Wind (0-100), Load (1000-5000)
X = torch.rand(N, 3) * torch.tensor([50, 100, 4000]) + torch.tensor([-10, 0, 1000])

# Target A: Instability (Classification)
# Unstable if (Temp > 35 AND Load > 4500) OR Wind > 90
stability_score = (X[:, 0] - 35) + (X[:, 2] - 4500)/100 + (X[:, 1] - 90) + torch.randn(N)*5
y_stable = (stability_score > 0).float().view(-1, 1)

# Target B: Demand (Regression)
# Demand driven by Temp (AC/Heating) and current Load
y_demand = (X[:, 2] * 1.1) + (X[:, 0].abs() * 20) + torch.randn(N) * 50
y_demand = y_demand.view(-1, 1).float()

print(f"Data Ready. X: {X.shape}, y_stable: {y_stable.shape}, y_demand: {y_demand.shape}")

Data Ready. X: torch.Size([100, 3]), y_stable: torch.Size([100, 1]), y_demand: torch.Size([100, 1])


## **Level 1: The Blackout Predictor (Deep Classification)**
**Your Task:** Detect Instability.

**Blueprint (The 'Funnel' Architecture):**
1.  **Input:** 3 Features
2.  **Layer 1:** 64 Neurons, `Tanh`
3.  **Layer 2:** 32 Neurons, `Tanh`
4.  **Layer 3:** 16 Neurons, `Tanh`
5.  **Output:** 1 Neuron, `Sigmoid`

**Training Specs:**
* Loss: `BCELoss`
* Optimizer: `SGD` (lr=0.01)
* Epochs: 200

In [ ]:
# LEVEL 1: WRITE YOUR CODE HERE

# 1. Define 'GridModel'
class GridModel(nn.Module):
    # TODO: Implement the 4-layer funnel
    def __init__(self):
      super(GridModel,self).__init__()
      self.fc1=nn.Linear(3,64)
      self.tanh=nn.Tanh()
      self.fc2=nn.Linear(64,32)
      self.tanh=nn.Tanh()
      self.fc3=nn.Linear(32,16)
      self.tanh=nn.Tanh()
      self.fc4=nn.Linear(16,1)
      self.sigmoid=nn.Sigmoid()
    def forward(self,x):
      x=self.fc1(x)
      x=self.tanh(x)
      x=self.fc2(x)
      x=self.tanh(x)
      x=self.fc3(x)
      x=self.tanh(x)
      x=self.fc4(x)
      x=self.sigmoid(x)
      return x

model_grid = GridModel()

# 2. Optimizer & Loss
loss_fn=nn.BCELoss()
optimizer=optim.SGD(model_grid.parameters(),lr=0.01,momentum=0.9)
# 3. Training Loop
epochs=200
train_loss=0
for epoch in range(epochs):
  model_grid.train()
  pred=model_grid(X)
  loss=loss_fn(pred,y_stable)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  train_loss+=loss.item()
# 4. Accuracy Check
y_pred=(pred>0.5).float()
accuracy=(y_pred==y_stable).float().sum()
if epoch%40==0:
  print(f"Accuracy: {accuracy}")

In [ ]:
# TEST LEVEL 1
try:
    # Check Layer Count
    layers = [m for m in model_grid.modules() if isinstance(m, nn.Linear)]
    assert len(layers) == 4, f"Blueprint requires 4 Linear layers. Found {len(layers)}"

    # Check Funnel Shapes
    assert layers[0].out_features == 64
    assert layers[1].out_features == 32
    assert layers[2].out_features == 16

    print("✅ Level 1 Passed: The Funnel Architecture is correct.")
except Exception as e:
    print(f"❌ Level 1 Fail: {e}")

✅ Level 1 Passed: The Funnel Architecture is correct.


## **Level 2: Demand Forecasting (Wide Regression)**
**Your Task:** Predict Demand (MW).

**Step 2.1: Pipeline**
* Split Train(20) / Test(80). `DataLoader` (Batch=10).

**Step 2.2: The 'Fat' Model (Overfitting Risk)**
* `Linear(3, 256) -> ReLU`
* `Linear(256, 256) -> ReLU`
* `Linear(256, 1)`

**Step 2.3: Diagnosis**
* Train for 300 epochs. Confirm Overfitting.

**Step 2.4: The Fix (Dropout)**
* Rebuild with `nn.Dropout(0.5)` after the first two ReLUs.
* Retrain.

In [ ]:
# LEVEL 2: WRITE YOUR CODE HERE
from sklearn.model_selection import train_test_split
# 1. Data Pipeline
X_train,X_test,y_train,y_test=train_test_split(X,y_stable,test_size=0.8)
train_ds=TensorDataset(X_train,y_train)
val_ds=TensorDataset(X_test,y_test)

train_loader=DataLoader(train_ds,batch_size=10,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=10)
# 2. Define 'model_fat' (256 width)
class ModelFat(nn.Module):
  def __init__(self):
    super(ModelFat,self).__init__()
    self.fc1=nn.Linear(3,256)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(256,256)
    self.fc3=nn.Linear(256,1)
  def forward(self,x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    x=self.relu(x)
    x=self.fc3(x)
    return x
model_fat=ModelFat()
loss_fn=nn.MSELoss()
# 3. Train Loop
epochs=300
for epoch in range(epochs):
  model_fat.train()
  train_loss=0
  val_loss=0
  for x_batch,y_batch in train_loader:
    pred=model_fat(x_batch)
    loss=loss_fn(pred,y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()
  train_loss/=len(train_loader)
  model_fat.eval()
  with torch.no_grad():
      pred=model_fat(x_batch)
      loss=loss_fn(pred,y_batch)
      val_loss+=loss.item()
  val_loss/=len(val_loader)
  if epoch%100==0:
    print(f"Epoch: {epoch} Training Loss: {train_loss} Test loss: {val_loss}")
# 4. Define 'model_smart' (With Dropout 0.5)
class ModelSmart(nn.Module):
    def __init__(self):
        super(ModelSmart, self).__init__()
        self.fc1 = nn.Linear(3, 256)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(p=0.5)

        self.fc2 = nn.Linear(256, 256)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(p=0.5)

        self.fc3 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)

        x = self.fc3(x)
        return x

model_smart = ModelSmart()
optimizer = torch.optim.Adam(model_fat.parameters(), lr=0.001)
# 5. Retrain
print("After dropout")
for epoch in range(epochs):
  train_loss=0
  val_loss=0
  model_smart.train()
  for x_batch,y_batch in train_loader:
    pred=model_smart(x_batch)
    loss=loss_fn(pred,y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()
  train_loss/=len(train_loader)
  model_smart.eval()
  with torch.no_grad():
        pred=model_smart(x_batch)
        loss=loss_fn(pred,y_batch)
        val_loss+=loss.item()
  val_loss/=len(val_loader)
  if epoch%100==0:
        print(f"Epoch :{epoch} Training Loss: {train_loss} Test Loss: {val_loss}")

Epoch: 0 Training Loss: 25354.189453125 Test loss: 3005.2255859375
Epoch: 100 Training Loss: 25354.19140625 Test loss: 2745.59423828125
Epoch: 200 Training Loss: 25354.189453125 Test loss: 3155.80078125
After dropout
Epoch :0 Training Loss: 176556.4140625 Test Loss: 8467.927734375
Epoch :100 Training Loss: 173856.44921875 Test Loss: 8176.77978515625
Epoch :200 Training Loss: 270466.0078125 Test Loss: 8275.36328125


In [ ]:
# TEST LEVEL 2
try:
    # Check Width
    l1 = list(model_smart.modules())[1]
    if isinstance(l1, nn.Sequential): l1 = l1[0]
    assert l1.out_features == 256, f"Hidden layer must be 256 wide. Found {l1.out_features}"

    # Check Dropout
    drop = [m for m in model_smart.modules() if isinstance(m, nn.Dropout)]
    assert len(drop) >= 2, "Missing dropout layers"
    assert drop[0].p == 0.5, "Dropout rate must be 0.5"

    print("✅ Level 2 Passed: Wide Architecture built correctly.")
except Exception as e:
    print(f"❌ Level 2 Fail: {e}")

✅ Level 2 Passed: Wide Architecture built correctly.


## **Level 3: The Architect (Manual MAE)**

**Part 3.1: Manual MAE (Mean Absolute Error)**
Implement `my_custom_mae`.
$$ Loss = Mean( |y_{pred} - y_{true}| ) $$
*Hint: Use `torch.abs()`.*

**Part 3.2: The Tournament**
Compare these 3 configs on **Demand Prediction**:

1.  **"Standard"**: `Linear(3, 64) -> ReLU -> Linear(64, 1)` (Loss: `MSELoss`)
2.  **"Leaky"**: `Linear(3, 64) -> LeakyReLU(0.1) -> Linear(64, 1)` (Loss: `L1Loss`)
3.  **"Mobile"**: `Linear(3, 64) -> Hardswish -> Linear(64, 1)` (Loss: **YOUR** `my_custom_mae`)

**Note:** `Hardswish` is a memory-efficient activation used in MobileNetV3.

In [ ]:
# LEVEL 3.1: MANUAL MAE
def my_custom_mae(pred, target):
    # TODO: Implement Mean Absolute Error
    loss=torch.mean(torch.abs(pred-target))
    return loss


# LEVEL 3.2: TOURNAMENT
experiments = [
    {"name": "Standard", "act": nn.ReLU(), "loss": nn.MSELoss()}, # ReLU, MSE
    {"name": "Leaky",    "act": nn.LeakyReLU(0.1), "loss": nn.L1Loss()}, # LeakyReLU(0.1), L1
    {"name": "Mobile",   "act": nn.Hardswish(), "loss": my_custom_mae}  # Hardswish, Custom MAE
]

print(f"{'Name':<10} | {'Test Loss':<10}")
print("-"*25)

# Loop...

Name       | Test Loss 
-------------------------


In [ ]:
# TEST LEVEL 3
try:
    # Test Math
    p = torch.tensor([-2.0]); t = torch.tensor([2.0])
    assert my_custom_mae(p, t).item() == 4.0, "Math Fail: |-2 - 2| should be 4"

    # Check Configs
    assert isinstance(experiments[2]['act'], nn.Hardswish), "Exp 3 Act must be Hardswish"
    assert not isinstance(experiments[2]['loss'], type), "Exp 3 Loss must be custom function"

    print("✅ Level 3 Passed: Manual MAE and Configs valid.")
except Exception as e:
    print(f"❌ Level 3 Fail: {e}")

✅ Level 3 Passed: Manual MAE and Configs valid.


## **Level 4: The Mechanic (Manual Optimization)**
**Your Task:** The `optim` library is deleted. Train a simple Linear Regression model manually.

1.  Create a single layer `nn.Linear(1, 1)`.
2.  Write a loop that updates weights using: `w = w - lr * gradient`.
3.  **Important:** You must use `with torch.no_grad():` for the update step.

In [ ]:
# LEVEL 4: THE PURGE (Run this to delete optimizers)
import torch.optim as optim
import gc
for var in list(locals().keys()):
    if 'opt' in var or 'optimizer' in var:
        del locals()[var]
gc.collect()
print("Optimizers deleted. You are on your own.")

Optimizers deleted. You are on your own.


In [ ]:
# LEVEL 4: WRITE YOUR CODE HERE

# 1. Setup Data
X_simple = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_simple = torch.tensor([[3.0], [5.0], [7.0], [9.0]]) # Target: y = 2x + 1

# 2. Define Model
model_manual = nn.Linear(1,1)
loss_fn=nn.MSELoss()
lr=0.01
epochs=200


# 3. Manual Loop
    # Forward, Loss, Backward
for epoch in range(epochs):
    pred=model_manual(X_simple)
    loss=loss_fn(pred,y_simple)
    loss.backward()
    # Update: param -= lr * param.grad
    with torch.no_grad():
      model_manual.weight-=lr*model_manual.weight.grad
      model_manual.bias-=lr*model_manual.bias.grad
    # Zero grad
    model_manual.weight.grad.zero_()
    model_manual.weight.grad.zero_()

    if epoch%50==0:
      print(f"Epoch: {epoch} Loss: {loss.item()}")

Epoch: 0 Loss: 37.68559646606445
Epoch: 50 Loss: 0.644736647605896
Epoch: 100 Loss: 0.02260703593492508
Epoch: 150 Loss: 0.0009334710775874555


In [ ]:
# TEST LEVEL 4
try:
    w = model_manual.weight.item()
    b = model_manual.bias.item()
    assert abs(w - 2.0) < 0.2, f"Convergence Fail: Weight is {w:.2f}, expected ~2.0"
    print("✅ Level 4 Passed: Manual Optimization successful!")
except Exception as e:
    print(f"❌ Level 4 Fail: {e}")

✅ Level 4 Passed: Manual Optimization successful!
